# Financial RAG ingestion on Colab: PDFs → Qdrant `docs_text` + `visual_pages`

Runs [scripts/rag/financial_ingest.py](../scripts/rag/financial_ingest.py), the PDF part of
[docs/financial_rag_final_design.md](../docs/financial_rag_final_design.md) (sections 5 and 7).
Full guide: [docs/financial_ingestion_colab.md](../docs/financial_ingestion_colab.md).

| Collection · vector | What | Where |
|---|---|---|
| `docs_text` · `dense` + `bm25` | text and table chunks; number-aware sparse terms (`391,035` = `391035`) | RAM |
| `visual_pages` · `colpali` | full ColPali 1030 × 128, float16, no HNSW (rerank only) | disk (mmap) |
| `visual_pages` · `mean_rows`, `mean_cols` | 38 × 128 pooled, float16, HNSW (first stage) | RAM |
| `visual_pages` · `summary` | text vector of the page text | RAM |

**Runtime:** GPU. An L4 is preferred (bfloat16). On a T4 the default is float16; if ColPali returns non-finite
values, rerun with `--colpali-dtype float32`.

**Cheaper split (recommended):**
1. Run `STAGES = "text"` on a **CPU** runtime (page analysis, OCR, text vectors).
2. Then run `STAGES = "visual"` on a **GPU** runtime. The page-analysis cache travels with the snapshots, so
   the GPU only encodes pages.

**Data:** the download cell fetches the EDGAR corpus to Drive. It takes ~7 minutes and ~1.8 GB (**measured**),
and resumes if interrupted:
- the 100 ARS PDFs;
- the matching 10-K iXBRL HTML;
- the DEF 14A proxy statements;
- companyfacts and submissions JSON.

**Rules:**
- Qdrant storage stays on `/content` (local disk), never on the Drive mount.
- PDFs, snapshots and the ingest state live on Drive.
- Everything is resumable. After a disconnect, rerun cells 1–5 and then the ingest cell; finished filings
  are skipped.

In [ ]:
# 1. Settings
DRIVE_ROOT = "/content/drive/MyDrive/capstone_rag"     # persistent: EDGAR corpus, snapshots, ingest state
WORK = "/content/rag"                                   # fast local disk: Qdrant storage and caches
REPO = "https://github.com/rama159065/Capstone.git"     # private repo: see the note in cell 3
BRANCH = "finance_report_qa_agent"
CODE_ZIP = "/content/drive/MyDrive/capstone_rag/capstone_code.zip"   # python -m scripts.common.colab_bundle, then upload to Drive
SEC_USER_AGENT = "capstone-research your.name@example.com"   # SEC requires a contact e-mail
STAGES = "text,visual"    # or "text" on a CPU runtime, then "visual" on a GPU runtime

In [ ]:
# 2. Drive, folders, GPU
import os
from google.colab import drive
drive.mount("/content/drive")
for path in (f"{DRIVE_ROOT}/edgar/pdfs", f"{DRIVE_ROOT}/snapshots", WORK):
    os.makedirs(path, exist_ok=True)
os.environ["RAG_INGEST_DIR"] = WORK          # --root default for every command below
os.environ["SEC_USER_AGENT"] = SEC_USER_AGENT
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU: run only the text stage"
!df -h /content | tail -1

In [ ]:
# 3. Code (the zip from python -m scripts.common.colab_bundle on Drive; else a clone of REPO
#    with the Colab secret GITHUB_TOKEN) and dependencies.
import json, subprocess, sys
from pathlib import Path


def fetch_code(root, repo_url, branch):
    """Unpack the code zip (Drive, else one you upload) into root; without a zip, clone the repo."""
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    uploaded = Path("/content") / Path(CODE_ZIP).name
    source = next((p for p in (Path(CODE_ZIP), uploaded) if p.exists()), None)
    if source is None and not root.exists():
        from google.colab import files
        print(f"{CODE_ZIP} not found: upload capstone_code.zip (cancel to clone {repo_url} instead)")
        names = list(files.upload())
        source = Path("/content") / names[0] if names else None
    if source is not None:
        with zipfile.ZipFile(source) as archive:
            info = archive.read("BUNDLE_INFO.json")
            stamp = root / "BUNDLE_INFO.json"
            if not stamp.exists() or stamp.read_bytes() != info:      # a new build: unpack over the old one
                archive.extractall(root)
        print("code:", source, json.loads(info))
    elif not root.exists():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")      # needed while the repo is private
        except Exception:
            token = None
        url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, str(root)], check=True)


fetch_code(Path("/content/Capstone"), REPO, BRANCH)
sys.path.insert(0, "/content/Capstone")
%cd /content/Capstone
!pip -q install "qdrant-client>=1.12,<2" "pymupdf>=1.24,<2" "transformers>=4.57,<5" "sentence-transformers>=3" "rapidocr>=3,<4" onnxruntime
# Optional GPU OCR: pip uninstall -y onnxruntime && pip install onnxruntime-gpu, then add --ocr-cuda

In [ ]:
# 4. Qdrant server on local disk, bound to localhost only
import subprocess, time, urllib.request
QDRANT_DIR = "/content/qdrant"
if not os.path.exists(f"{QDRANT_DIR}/qdrant"):
    os.makedirs(QDRANT_DIR, exist_ok=True)
    # For the final runs, pin a tested version: .../releases/download/vX.Y.Z/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !curl -sL https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C {QDRANT_DIR}
env = {**os.environ, "QDRANT__SERVICE__HOST": "127.0.0.1",
       "QDRANT__STORAGE__STORAGE_PATH": f"{WORK}/qdrant",
       "QDRANT__STORAGE__SNAPSHOTS_PATH": f"{WORK}/snapshots",   # = <root>/snapshots, used by restore
       "QDRANT__TELEMETRY_DISABLED": "true"}
try:
    urllib.request.urlopen("http://127.0.0.1:6333/readyz", timeout=2)
    print("Qdrant already running")
except Exception:
    qdrant = subprocess.Popen([f"{QDRANT_DIR}/qdrant"], cwd=QDRANT_DIR, env=env,
                              stdout=open(f"{WORK}/qdrant.log", "ab"), stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            urllib.request.urlopen("http://127.0.0.1:6333/readyz", timeout=2)
            break
        except Exception:
            time.sleep(1)
print(urllib.request.urlopen("http://127.0.0.1:6333").read().decode())
os.environ["QDRANT_URL"] = "http://127.0.0.1:6333"

In [ ]:
# 5. Restore the previous session's snapshots and ingest state (nothing to do on the first session)
import glob
if glob.glob(f"{DRIVE_ROOT}/snapshots/*.snapshot") and not os.path.exists(f"{WORK}/ingest.sqlite3"):
    !python -m scripts.rag.financial_ingest restore --snapshot-dir {DRIVE_ROOT}/snapshots
else:
    print("nothing to restore (first session, or this runtime already has the index)")

## Download the corpus (EDGAR → Drive)

Fetched from SEC EDGAR, one row per filing in `corpus_manifest.jsonl` (linked by `filing_id`):

| Folder | Content |
|---|---|
| `pdfs/` | ARS PDFs, SHA-256 checked against the manifest |
| `10k/` | 10-K iXBRL HTML, plus sibling iXBRL documents when the statements are split out (Wells Fargo) |
| `proxy/` | DEF 14A proxy statements |
| `companyfacts/`, `submissions/` | SEC JSON, once per company |

Rerunning it fetches only what is missing.

In [ ]:
!python -m scripts.downloads.download_edgar_corpus --output-dir {DRIVE_ROOT}/edgar --user-agent "{SEC_USER_AGENT}"
!du -sh {DRIVE_ROOT}/edgar/*

## Optional: the sector corpus (1,004 annual reports, 13 sectors)

> **For the full 1,004-filing corpus use [colab_rag_pipelines.ipynb](colab_rag_pipelines.ipynb)**: it downloads, ingests sector by sector, reports coverage of the plan and serves the index, with the folder layout in [docs/annual_report_rag_structure.md](../docs/annual_report_rag_structure.md). The cells below add sectors to *this* notebook's index (`/content/rag`, snapshots in `snapshots/`), which is a separate index.

The plan in [docs/annual_report_corpus_plan.html](../docs/annual_report_corpus_plan.html), fetched from EDGAR by
[scripts/downloads/download_annual_report_corpus.py](../scripts/downloads/download_annual_report_corpus.py)
into one Drive folder per sector:

```
annual_reports/
  pdf_manifest.jsonl            all sectors (local_path = banking/EDGAR_ARS_JPM_2025.pdf)
  download_summary.json
  banking/pdf_manifest.jsonl    this sector only
  banking/EDGAR_ARS_JPM_2025.pdf
  oil_and_gas/...
```

- **Size:** all 13 sectors are 11.3 GB, so check your free Drive space first. The dry run prints the size of what you selected.
- **Collections:** the same `docs_text` / `visual_pages` collections. Every point gets a payload-indexed
  `sector` / `sector_slug` field, so `query --sector banking` narrows to one sector.
- **Reruns:** reruns are resumable, like the EDGAR download above. Failed PDFs are retried.

In [ ]:
SECTORS = "banking,insurance,oil_and_gas"   # names or slugs, comma-separated; "" = all 13 sectors
MAX_PER_SECTOR = 20                          # largest companies first; None = every planned PDF of those sectors
ANNUAL = f"{DRIVE_ROOT}/annual_reports"
SELECT = (f"--sectors {SECTORS} " if SECTORS else "") + (f"--max-per-sector {MAX_PER_SECTOR}" if MAX_PER_SECTOR else "")
!python -m scripts.downloads.download_annual_report_corpus --dry-run {SELECT}

In [ ]:
!python -m scripts.downloads.download_annual_report_corpus --output-dir {ANNUAL} --user-agent "{SEC_USER_AGENT}" {SELECT}
!du -sh {ANNUAL}/*/ | sort -k2

In [ ]:
# Ingest the downloaded sectors. Filings already ingested from the EDGAR corpus (same document_id) are skipped;
# tag-sectors then writes the sector field onto their points without re-embedding.
SECTOR_ARG = f"--sectors {SECTORS}" if SECTORS else ""
!python -m scripts.rag.financial_ingest ingest --manifest {ANNUAL} {SECTOR_ARG} --stages {STAGES} \
    --snapshot-dir {DRIVE_ROOT}/snapshots --batch-size 10 --workers 2 --colpali-batch 4 2>&1 | grep -v pymupdf_layout
!python -m scripts.rag.financial_ingest tag-sectors --manifest {ANNUAL} {SECTOR_ARG}
!python -m scripts.rag.financial_ingest query "net interest income" --sector banking

## Smoke test (embedded Qdrant, separate folder, one filing, 8 pages)

Checks the download, page analysis, OCR, ColPali model and pooling before the long run. It does not touch the
server collections.

In [ ]:
!QDRANT_URL= python -m scripts.rag.financial_ingest ingest --root /content/rag_smoke --stages {STAGES} \
    --pdf-dir {DRIVE_ROOT}/edgar/pdfs --download-missing --limit-filings 1 --max-pages-per-pdf 8 --reset 2>&1 | grep -v pymupdf_layout
!QDRANT_URL= python -m scripts.rag.financial_ingest status --root /content/rag_smoke

## Full ingest

Snapshots go to Drive after every batch of 10 filings. Each is checksum-verified, and only the newest is kept.
Progress is one JSON line per filing and stage.

**For 100 filings (14.7k pages):**
- **Text stage:** ~0.5 s/page/core (**measured** locally), so ~1 h on 2 workers.
- **Visual stage:** ~40% of pages. **Measured:** 24–27% for clean filings, up to 95% for a garbled one.
  That is ~5,900 pages at ~0.5–1.5 pages/s on a T4 (**estimate**), so ~1–3.5 GPU-hours.

In [ ]:
!python -m scripts.rag.financial_ingest ingest --stages {STAGES} --pdf-dir {DRIVE_ROOT}/edgar/pdfs --download-missing \
    --snapshot-dir {DRIVE_ROOT}/snapshots --batch-size 10 --workers 2 --colpali-batch 4 2>&1 | grep -v pymupdf_layout

In [ ]:
# Status: points per collection, filings per stage, failures (rerun the ingest cell to retry them)
!python -m scripts.rag.financial_ingest status
!du -sh {WORK}/qdrant {DRIVE_ROOT}/snapshots

In [ ]:
# Retrieval checks: exact-number text search, then the visual path (pooled prefetch -> full MaxSim rerank)
!python -m scripts.rag.financial_ingest query "total revenue 130,497" --ticker NVDA
!python -m scripts.rag.financial_ingest query "revenue by market platform bar chart" --ticker NVDA --visual

## Pipeline 2: OCR / VLM transcription → Chroma

The cells above are **pipeline 1**:
- ColPali page vectors in Qdrant `visual_pages`: mean-pooled rows and columns in RAM, full multivectors on disk.
- To send only table and chart pages to ColPali, add `--visual-pages tables_charts` to the ingest command. The default also includes pictures and unreadable pages.

**Pipeline 2** ([scripts/rag/vlm_ingest.py](../scripts/rag/vlm_ingest.py)) turns pages into text first:
- **Visual pages:** the page reader (`READER`) turns table, chart and picture pages into Markdown. `ocr` = RapidOCR + native tables (CPU, free); `transformers` = local Qwen3-VL; `openai` = an OpenAI-compatible endpoint, self-hosted Qwen-VL or a paid API (set `RAG_VISION_API_BASE`, `RAG_VISION_MODEL`, `RAG_VISION_API_KEY`). Tables become Markdown tables, and charts become one line per labelled value.
- **Other pages:** plain narrative pages keep their text layer.
- **Index (Chroma only):** chunks are embedded into Chroma, and a BM25 sidecar (`chroma/bm25.sqlite3`) keeps exact figures searchable; retrieval fuses both by RRF. Qdrant is pipeline 1's database. Each chunk carries the same `sector`, `ticker`, `fiscal_year` and citable `chunk_id` as pipeline 1.
- **Checked numbers:** figures the VLM wrote that the page's text layer doesn't print are kept as `unverified_numbers`.

**Runtime:**
- Use a GPU. On a T4, Qwen3-VL-4B in 4-bit takes roughly 15–40 s per page. This is an **estimate**; `status` prints the measured rate.
- Transcriptions are cached on Drive page by page, so a disconnect loses one page, and `--force` re-indexes without calling the VLM again.

In [ ]:
!pip -q install "chromadb>=1.1" "bitsandbytes>=0.48" accelerate
PIPELINE2 = f"{DRIVE_ROOT}/vlm"                 # Chroma index + state + transcription cache, all on Drive
READER = "transformers"                         # "ocr" (CPU) / "transformers" (Qwen-VL) / "openai" (paid or self-hosted API)
VLM_MODEL = "Qwen/Qwen3-VL-4B-Instruct"         # Qwen/Qwen3-VL-2B-Instruct (faster) or -8B on an L4/A100
VLM_SOURCE = f"--manifest {ANNUAL} {SECTOR_ARG}" if "ANNUAL" in globals() else f"--pdf-dir {DRIVE_ROOT}/edgar/pdfs"

In [ ]:
# Smoke test first: one filing, at most 3 VLM pages
!python -m scripts.rag.vlm_ingest ingest --root /content/vlm_smoke --store chroma {VLM_SOURCE} --limit-filings 1 \
    --max-pages-per-pdf 20 --vlm-max-pages-per-pdf 3 --vlm transformers --vlm-model {VLM_MODEL} --reset 2>&1 | grep -v pymupdf_layout
!python -m scripts.rag.vlm_ingest status --root /content/vlm_smoke

In [ ]:
!python -m scripts.rag.vlm_ingest ingest --root {PIPELINE2} {VLM_SOURCE} \
    --vlm {READER} {MODEL_ARG} --vlm-pages visual 2>&1 | grep -v pymupdf_layout
!python -m scripts.rag.vlm_ingest status --root {PIPELINE2}
!python -m scripts.rag.vlm_ingest query "net interest income" --root {PIPELINE2} --sector banking

## Serve: FastAPI + Streamlit on this Colab runtime

The API, the UI and Qdrant run on one VM:

```
browser → Colab port proxy (only you, signed in) → Streamlit :8501 → FastAPI :8000 → Qdrant :6333 (/content/rag)
```

**What the runtime profile sets** (`RAG_PROFILE=colab`, [scripts/common/runtime_profile.py](../scripts/common/runtime_profile.py)):
- `QDRANT_URL=http://127.0.0.1:6333` and `RAG_INGEST_DIR=/content/rag`;
- financial retrieval from the ingested collections (`RAG_BACKEND=financial_qdrant`) as soon as `ingest.sqlite3` exists, i.e. after the ingest or the restore cell;
- `RAG_DEVICE=cuda` on a GPU runtime.

**Before running:**
- Run cells 1–5 first. Ingest, or restore the snapshots, before serving.
- A CPU runtime is enough for serving text retrieval.
- For ColPali visual retrieval, set `RAG_FIN_VISUAL=1`, preferably on a GPU runtime: a query takes ~2.5 s on CPU.
- For the provided LLM, add `LLM_API_BASE`, `LLM_API_KEY` and `LLM_MODEL` as Colab secrets, then pick `hosted` in the Streamlit sidebar. Without them, only `mock` works.

In [ ]:
# Serving dependencies (FastAPI, Streamlit, LangGraph, Chroma...) from pyproject.toml. The repo is not an
# installable package (`pip install -e .` fails), so install its dependency list. Keep Colab's CUDA torch.
import re, subprocess, sys, tomllib
deps = tomllib.load(open("/content/Capstone/pyproject.toml", "rb"))["project"]["dependencies"]
deps = [d for d in deps if re.split(r"[<>=!~ \[]", d)[0] not in {"torch", "torchvision", "jupyterlab"}]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *deps])
from google.colab import userdata
for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"{key} is not a Colab secret: use llm_provider=mock, or add the secret and rerun this cell")
os.environ["RAG_PROFILE"] = "colab"
# os.environ["RAG_FIN_VISUAL"] = "1"     # ColPali visual retrieval as well (loads the model on first query)

In [ ]:
# Start FastAPI (one worker: conversations and caches live in that process) and Streamlit, both on localhost.
import subprocess, time, urllib.request, json
def start(name, args, log):
    return subprocess.Popen(args, cwd="/content/Capstone", env=os.environ.copy(),
                            stdout=open(f"{WORK}/{log}", "ab"), stderr=subprocess.STDOUT)
api = start("api", [sys.executable, "-m", "uvicorn", "scripts.api.api_app:app", "--host", "127.0.0.1",
                    "--port", "8000", "--workers", "1"], "api.log")
for _ in range(120):
    try:
        health = json.load(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"API did not start; see {WORK}/api.log")
print("API runtime:", health["runtime"])     # expect profile=colab, financial_backend=financial_qdrant
ui = start("ui", [sys.executable, "-m", "streamlit", "run", "scripts/ui/api_streamlit_app.py",
                  "--server.port", "8501", "--server.address", "127.0.0.1", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"], "streamlit.log")
time.sleep(8)
from google.colab.output import eval_js
print("Open the chat UI:", eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
# Quick API check without the UI: one clarification round, then an answer from the Qdrant collections
def turn(query, cid=None):
    body = json.dumps({"query": query, "conversation_id": cid}).encode()
    request = urllib.request.Request("http://127.0.0.1:8000/conversation/financial_report_qa", data=body,
                                     headers={"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(request, timeout=300))
first = turn("What was NVIDIA's revenue?")
print(first["status"], "|", first.get("message"))
second = turn("FY2025", first["conversation_id"])
print(second["status"], "|", second.get("resolved_query"), "|", second["result"]["final_status"])
print([e["chunk_id"] for e in second["result"].get("evidence_preview", [])][:3])

In [ ]:
# Stop serving (Qdrant keeps running for further ingest or restore)
for process in (ui, api):
    process.terminate()

## Next

- **Serving on your PC instead:** with no Colab detected, the same code uses the `local` profile (embedded Qdrant in
  `indexes/financial_qdrant`, or set `QDRANT_URL`). See docs/financial_ingestion_colab.md, section 7.
- **Facts DB:** this notebook does not write the 10-K HTML / XBRL facts (store A); that is a separate step.